# Presidential elections 2002 & 2022 — first-round results by département

Scrapes the official Ministère de l'Intérieur archive
(`archives-resultats-elections.interieur.gouv.fr`) for the **first round** of the 2002 and 2022
presidential elections at **département** level.

**Approach**

1. Start from the national results page of each election.
2. Crawl the links on that page (and, if needed, on the region pages it links to) and recognise
   département pages from the link label (e.g. `Ain`, `Ain (01)`) or, as a fallback, from a
   `region/département` URL pattern (e.g. `.../084/001/index.php`). No département URL is hard-coded.
3. Visit each département page, find the first-round tables (turnout table + candidate table) and
   parse them.
4. Candidate-level output is in long format: one row per `year × département × candidate`.

**Notes for debugging**

* The parsers are written against the usual structure of these pages (HTML `<table>`s with
  `Inscrits / Abstentions / Votants / Blancs / Nuls / Exprimés` rows and a candidate table with
  `Voix` and `% Exprimés` columns). The live HTML was **not** inspected when writing this notebook, so
  use `show_links(...)` and `describe_tables(...)` (defined in the helpers) to look at a page if something
  does not match.
* Downloaded pages are cached in `data/raw/html_cache/`, so re-running cells does not hit the site again.
  Set `USE_CACHE = False` to force fresh downloads.
* Set `SCRAPE_LIMIT = 5` to test the loops on a few départements before the full run.

## 1. Imports

In [101]:
import re
import time
import unicodedata
from pathlib import Path
from urllib.parse import urldefrag, urljoin, urlparse

import pandas as pd
import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

## 2. URLs and configuration

In [102]:
START_URLS = {
    2002: "https://www.archives-resultats-elections.interieur.gouv.fr/resultats/presidentielle_2002/index.php",
    2022: "https://www.archives-resultats-elections.interieur.gouv.fr/resultats/presidentielle-2022/index.php",
}
SITE_HOME = "https://www.archives-resultats-elections.interieur.gouv.fr/"

# Number of first-round candidates (the second round always has exactly 2)
N_CANDIDATES = {2002: 16, 2022: 12}

# Candidates who ran only in the first round: finding them on every page proves we parsed round 1
FIRST_ROUND_ONLY = {2002: ["JOSPIN", "BAYROU"], 2022: ["ZEMMOUR", "PECRESSE"]}

RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
CACHE_DIR = RAW_DIR / "html_cache"  # raw HTML pages, so re-runs don't download again
USE_CACHE = True

REQUEST_DELAY = 1.0  # seconds between requests (be polite to the server)
TIMEOUT = 30

# The site answers 403 to non-browser clients: send the headers a normal browser sends.
HEADERS = {
    "User-Agent": ("Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
                   "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "fr-FR,fr;q=0.9,en-US;q=0.8,en;q=0.7",
    "Accept-Encoding": "gzip, deflate",
    "Connection": "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Referer": SITE_HOME,
}

# Set to e.g. 5 to test the scraping loops on a few départements first; None = all
SCRAPE_LIMIT = None

for d in (RAW_DIR, PROCESSED_DIR, CACHE_DIR):
    d.mkdir(parents=True, exist_ok=True)

### Reference list of départements

Used only to **recognise** département links and to **check for missing** départements — the URLs
themselves are discovered by crawling. Codes are INSEE codes (`01`…`95`, `2A`, `2B`, `971`…`976`);
other overseas collectivités and French voters abroad are kept with their own codes and flagged in `dep_type`.

In [103]:
METRO = """01;Ain
02;Aisne
03;Allier
04;Alpes-de-Haute-Provence
05;Hautes-Alpes
06;Alpes-Maritimes
07;Ardèche
08;Ardennes
09;Ariège
10;Aube
11;Aude
12;Aveyron
13;Bouches-du-Rhône
14;Calvados
15;Cantal
16;Charente
17;Charente-Maritime
18;Cher
19;Corrèze
2A;Corse-du-Sud
2B;Haute-Corse
21;Côte-d'Or
22;Côtes-d'Armor
23;Creuse
24;Dordogne
25;Doubs
26;Drôme
27;Eure
28;Eure-et-Loir
29;Finistère
30;Gard
31;Haute-Garonne
32;Gers
33;Gironde
34;Hérault
35;Ille-et-Vilaine
36;Indre
37;Indre-et-Loire
38;Isère
39;Jura
40;Landes
41;Loir-et-Cher
42;Loire
43;Haute-Loire
44;Loire-Atlantique
45;Loiret
46;Lot
47;Lot-et-Garonne
48;Lozère
49;Maine-et-Loire
50;Manche
51;Marne
52;Haute-Marne
53;Mayenne
54;Meurthe-et-Moselle
55;Meuse
56;Morbihan
57;Moselle
58;Nièvre
59;Nord
60;Oise
61;Orne
62;Pas-de-Calais
63;Puy-de-Dôme
64;Pyrénées-Atlantiques
65;Hautes-Pyrénées
66;Pyrénées-Orientales
67;Bas-Rhin
68;Haut-Rhin
69;Rhône
70;Haute-Saône
71;Saône-et-Loire
72;Sarthe
73;Savoie
74;Haute-Savoie
75;Paris
76;Seine-Maritime
77;Seine-et-Marne
78;Yvelines
79;Deux-Sèvres
80;Somme
81;Tarn
82;Tarn-et-Garonne
83;Var
84;Vaucluse
85;Vendée
86;Vienne
87;Haute-Vienne
88;Vosges
89;Yonne
90;Territoire de Belfort
91;Essonne
92;Hauts-de-Seine
93;Seine-Saint-Denis
94;Val-de-Marne
95;Val-d'Oise"""

OUTRE_MER = [
    ("971", "Guadeloupe", "DOM"),
    ("972", "Martinique", "DOM"),
    ("973", "Guyane", "DOM"),
    ("974", "La Réunion", "DOM"),
    ("976", "Mayotte", "DOM"),  # DOM since 2011 (collectivité départementale in 2002)
    ("975", "Saint-Pierre-et-Miquelon", "COM"),
    ("986", "Wallis-et-Futuna", "COM"),
    ("987", "Polynésie française", "COM"),
    ("988", "Nouvelle-Calédonie", "COM"),
    ("ZX", "Saint-Martin/Saint-Barthélemy", "COM"),
    ("ZZ", "Français établis hors de France", "etranger"),
]

DEPARTEMENTS = pd.DataFrame(
    [(*line.split(";"), "metropole") for line in METRO.splitlines()] + OUTRE_MER,
    columns=["dep_code", "dep_name", "dep_type"],
)

# Départements in the strict sense for each election (used by the "missing départements" check)
_strict = DEPARTEMENTS.loc[DEPARTEMENTS["dep_type"].isin(["metropole", "DOM"]), "dep_code"]
EXPECTED_DEPS = {
    2002: set(_strict) - {"976"},  # Mayotte was not yet a département in 2002
    2022: set(_strict),
}

# Letter codes used by the Ministère de l'Intérieur for overseas territories (in URLs / files)
MININT_CODES = {
    "ZA": "971", "ZB": "972", "ZC": "973", "ZD": "974", "ZM": "976", "ZS": "975",
    "ZW": "986", "ZP": "987", "ZN": "988", "ZX": "ZX", "ZZ": "ZZ", "099": "ZZ",
}

# Alternative spellings that may be used in link labels
NAME_ALIASES = {
    "Réunion": "974",
    "Guyane française": "973",
    "Côtes-du-Nord": "22",
    "Français de l'étranger": "ZZ",
    "Français établis hors de France": "ZZ",
    "Étranger": "ZZ",
    "Saint-Martin et Saint-Barthélemy": "ZX",
    "Saint-Barthélemy et Saint-Martin": "ZX",
    "Saint-Barthélemy/Saint-Martin": "ZX",
}

print(len(DEPARTEMENTS), "entities in the reference table")
print({year: len(codes) for year, codes in EXPECTED_DEPS.items()}, "départements expected (strict sense)")

107 entities in the reference table
{2002: 100, 2022: 101} départements expected (strict sense)


## 3. Helper functions

### Text and number cleaning

In [104]:
def strip_accents(text):
    return "".join(c for c in unicodedata.normalize("NFKD", str(text)) if not unicodedata.combining(c))


def norm_text(text):
    """Lowercase, no accents, punctuation -> spaces, collapsed whitespace."""
    text = strip_accents(text).lower()
    return re.sub(r"[^a-z0-9]+", " ", text).strip()


def norm_dep_name(text):
    """Normalise a link label so it can be matched to a département name.

    'Ain (01)', '01 - Ain', 'Département de l'Ain' -> 'ain'
    """
    text = re.sub(r"\(.*?\)", " ", str(text))
    key = norm_text(text)
    key = re.sub(r"^departement( de la| de l| du| des| de| d)? ", "", key)
    key = re.sub(r"^(0?\d{2,3}|0?2[ab]|z[a-z]) ", "", key)  # leading code
    key = re.sub(r" (0?\d{2,3}|0?2[ab]|z[a-z])$", "", key)  # trailing code
    return key


NAME_TO_CODE = {norm_dep_name(name): code for code, name in zip(DEPARTEMENTS["dep_code"], DEPARTEMENTS["dep_name"])}
NAME_TO_CODE.update({norm_dep_name(name): code for name, code in NAME_ALIASES.items()})


def to_int(text):
    """'1 234 567' (any kind of space, incl. non-breaking) -> 1234567. None if not an integer."""
    s = re.sub(r"\s", "", str(text))
    if re.fullmatch(r"\d{1,3}(\.\d{3})+", s):  # '1.234.567'
        s = s.replace(".", "")
    return int(s) if re.fullmatch(r"\d+", s) else None


def to_float(text):
    """'24,01 %' -> 24.01. None if not a number."""
    s = re.sub(r"\s", "", str(text)).replace("%", "").replace(",", ".")
    try:
        return float(s)
    except ValueError:
        return None

### HTTP (with retries and an on-disk cache)

In [105]:
try:
    # Optional (`pip install curl_cffi`): requests with a real browser TLS fingerprint.
    # Used automatically if plain `requests` is still refused with 403.
    from curl_cffi import requests as cffi_requests
except ImportError:
    cffi_requests = None


def make_session():
    session = requests.Session()
    session.headers.update(HEADERS)
    retry = Retry(total=3, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504])
    session.mount("https://", HTTPAdapter(max_retries=retry))
    session.mount("http://", HTTPAdapter(max_retries=retry))
    try:
        session.get(SITE_HOME, timeout=TIMEOUT)  # pick up any cookies set by the home page
    except requests.RequestException as e:
        print(f"Home page not reachable: {e!r}")
    return session


SESSION = make_session()


def cache_path(url):
    p = urlparse(url)
    name = p.netloc + p.path + ("?" + p.query if p.query else "")
    return CACHE_DIR / re.sub(r"[^A-Za-z0-9._-]+", "_", name)


def download(url):
    """GET a page with `requests`; if refused (403), retry with curl_cffi impersonating Chrome."""
    resp = SESSION.get(url, timeout=TIMEOUT)
    if resp.status_code == 403 and cffi_requests is not None:
        resp = cffi_requests.get(url, impersonate="chrome", timeout=TIMEOUT,
                                 headers={"Accept-Language": HEADERS["Accept-Language"], "Referer": SITE_HOME})
    if resp.status_code == 403:
        hint = ("install curl_cffi (`pip install curl_cffi`) and re-run the helper cells"
                if cffi_requests is None else "the site also refuses curl_cffi")
        raise requests.HTTPError(f"403 Forbidden for {url} — the server rejects this client; {hint}")
    resp.raise_for_status()
    return resp.content


def fetch(url):
    """Return the raw bytes of a page (from the cache if available). Errors are never cached."""
    path = cache_path(url)
    if USE_CACHE and path.exists():
        return path.read_bytes()
    content = download(url)
    time.sleep(REQUEST_DELAY)
    if USE_CACHE:
        path.write_bytes(content)
    return content


def get_soup(url):
    # Passing bytes lets BeautifulSoup detect the encoding declared in the page (<meta charset>),
    # which keeps accents correct for both UTF-8 and ISO-8859-1 pages.
    return BeautifulSoup(fetch(url), "html.parser")


def page_heading(soup):
    """Title of the results, e.g. 'PARIS (75) (résultats officiels)'.

    The <h1>/<title> of the archive pages are generic, so look first for a heading or <strong>
    containing a code in parentheses, like '(75)' or '(2A)'.
    """
    for tag in soup.find_all(["strong", "h1", "h2", "h3"]):
        text = " ".join(tag.get_text(" ", strip=True).split())
        if re.search(r"\(\s*(\d{2,3}|2[AB]|Z[A-Z])\s*\)", text):
            return text
    tag = soup.find("title")
    return tag.get_text(" ", strip=True) if tag else ""

### Link discovery

Département links may be ordinary `<a>` links, `<area>` elements of an image map, `<a xlink:href>`
inside an SVG map, `data-*` attributes, `onclick` handlers or `<option>`s of a drop-down list.
`iter_links` collects all of them together with a label (link text, `title`, `alt`, …).

In [106]:
LINK_ATTRS = ["href", "xlink:href", "data-href", "data-url", "data-link"]


def iter_links(soup, base_url):
    """Yield (absolute_url, label) for every link-like element of the page."""
    for tag in soup.find_all(True):
        target = next((tag.get(a) for a in LINK_ATTRS if tag.get(a)), None)
        if target is None and tag.get("onclick"):
            m = re.search(r"['\"]([^'\"]+\.(?:php|html?)[^'\"]*)['\"]", tag["onclick"])
            target = m.group(1) if m else None
        if target is None and tag.name == "option" and re.search(r"\.(php|html?)|/", tag.get("value", "")):
            target = tag["value"]
        if not target or target.startswith(("#", "javascript:", "mailto:")):
            continue
        url = urldefrag(urljoin(base_url, target.strip()))[0]
        label = (tag.get_text(" ", strip=True) or tag.get("title") or tag.get("alt")
                 or tag.get("aria-label") or tag.get("data-name") or "")
        yield url, label


def show_links(url, pattern=None):
    """Debugging: print every link found on a page (optionally only those whose URL/label matches `pattern`)."""
    rows = pd.DataFrame(list(iter_links(get_soup(url), url)), columns=["url", "label"]).drop_duplicates()
    if pattern:
        rows = rows[rows["url"].str.contains(pattern, case=False) | rows["label"].str.contains(pattern, case=False)]
    with pd.option_context("display.max_rows", 500, "display.max_colwidth", 120):
        display(rows)


CODE_SEG_RE = re.compile(r"\d{2,3}|0?2[AB]|Z[A-Z]", re.I)


def code_segments(url, root):
    """For .../<root>/084/001/index.php return ['084', '001']; None if the URL is not of that form."""
    rel = url[len(root):].split("?")[0]
    parts = [p for p in rel.split("/") if p]
    if not parts or not re.fullmatch(r"index\.(php|html?)", parts[-1]):
        return None
    parts = parts[:-1]
    return parts if parts and all(CODE_SEG_RE.fullmatch(p) for p in parts) else None


def code_from_url(url):
    """Guess the département code from the last code-like URL segment.

    .../084/001/index.php -> '01', .../02A/index.php -> '2A', .../ZA/index.php -> '971'
    """
    parts = [p for p in urlparse(url).path.split("/") if p]
    if parts and re.match(r"index\.", parts[-1]):
        parts = parts[:-1]
    elif parts:
        parts[-1] = parts[-1].rsplit(".", 1)[0]
    for seg in reversed(parts):
        seg = seg.upper()
        if seg in MININT_CODES:
            return MININT_CODES[seg]
        m = re.fullmatch(r"0?(\d[\dAB])", seg)
        if m:
            return m.group(1)
        if re.fullmatch(r"9[78]\d", seg):
            return seg
    return None

In [107]:
def discover_departement_urls(year, max_depth=2, max_pages=300):
    """Crawl from the national page and return one URL per département.

    - A link whose label is a département name (e.g. 'Ain', 'Ain (01)') is a département link.
    - Fallback: a link of the form <root>/<region>/<dep>/index.php is a département link.
    - Other index pages below the election root (typically region pages) are explored,
      down to `max_depth` clicks from the national page.
    Département pages themselves are never explored (their links point to communes).
    """
    start_url = START_URLS[year]
    root = start_url.rsplit("/", 1)[0] + "/"
    found, visited, queue = [], set(), [(start_url, 0)]

    while queue and len(visited) < max_pages:
        url, depth = queue.pop(0)
        if url in visited:
            continue
        visited.add(url)
        try:
            soup = get_soup(url)
        except Exception as e:
            print(f"  could not fetch {url}: {e!r}")
            continue

        for link, label in iter_links(soup, url):
            if not link.startswith(root) or link in visited:
                continue
            segs = code_segments(link, root)
            code, method = NAME_TO_CODE.get(norm_dep_name(label)), "link label"
            if code is None and segs and len(segs) == 2:
                code, method = code_from_url(link), "url pattern"
            if code:
                found.append({"dep_code": code, "url": link, "link_label": label, "method": method,
                              "found_on": url, "path_depth": len(urlparse(link).path.strip("/").split("/"))})
            elif segs and depth < max_depth:
                queue.append((link, depth + 1))

    print(f"{year}: visited {len(visited)} pages, {len(found)} département links found")
    if not found:
        print("No département link recognised. Inspect the start page with show_links(START_URLS[year]).")
        return pd.DataFrame(columns=["year", "dep_code", "dep_name", "dep_type", "url", "link_label", "method", "found_on"])

    links = pd.DataFrame(found)
    # Several links can point to the same département (map + list, region page + département page, ...).
    # Prefer: not a 2nd-round URL, matched by label, deepest URL (département page rather than region page).
    links["round2_hint"] = links["url"].str.contains(r"tour\W?2|/t2/|2nd|second", case=False, regex=True)
    links["by_label"] = links["method"].eq("link label")
    links = (links.sort_values(["dep_code", "round2_hint", "by_label", "path_depth"],
                               ascending=[True, True, False, False], kind="stable")
                  .drop_duplicates("dep_code"))

    links = links.merge(DEPARTEMENTS, on="dep_code", how="left")
    links.insert(0, "year", year)
    links = links[["year", "dep_code", "dep_name", "dep_type", "url", "link_label", "method", "found_on"]]

    missing = sorted(EXPECTED_DEPS[year] - set(links["dep_code"]))
    print(f"{year}: {len(links)} distinct départements/territories; missing (strict sense): {missing or 'none'}")
    return links.reset_index(drop=True)

### Parsing a results page

Structure observed on the 2002 pages (e.g. Paris, `.../presidentielle_2002/011/075/1175.php`): one outer
layout table wrapping small `fr-table` tables, **round 2 first**, then *"RAPPEL DES RESULTATS 1er tour"*:

| tables | content |
|---|---|
| 1 | `Nombre / % Inscrits`: Inscrits, Abstentions, Votants (round 2) |
| 2 | `Nombre / % Votants`: Blancs ou nuls, Exprimés (round 2) |
| 3 | `Voix / % Exprimés`: 2 candidates (round 2) |
| 4–6 | the same three tables for round 1 (16 candidates) |

The turnout figures are therefore **split over several tables**. The parser:

1. reads every (non-layout) `<table>` into rows of cell texts;
2. recognises **candidate tables** (a header cell `Voix`) and **turnout tables** (rows `Inscrits`, `Votants`, `Exprimés`, …);
3. keeps the candidate table with **more than 2 candidates** — the second round always has exactly 2;
4. merges the turnout tables printed just **above** it (between the previous candidate table and this one)
   and, separately, those printed just **below** it (between this one and the next candidate table),
   and keeps the block whose `Exprimés` equals the sum of the candidates' votes.

The 2022 pages (e.g. Ain, `.../presidentielle-2022/084/001/index.php`) use the other order: round-2
candidates, round-2 turnout, then *"Rappel des résultats du département au 1er tour"*: round-1 candidates,
round-1 turnout. There the round-1 turnout is **below** the candidates, and the block above them is the
round-2 turnout — hence the check against the sum of votes.

If no first-round table is on the page but a link labelled `1er tour` exists, that link is followed.

In [108]:
TURNOUT_FIELDS = [  # order matters: 'blancs ou nuls' before 'blancs'
    ("blancs ou nuls", "blancs_nuls"),
    ("blancs et nuls", "blancs_nuls"),
    ("blancs", "blancs"),
    ("nuls", "nuls"),
    ("inscrits", "inscrits"),
    ("abstention", "abstentions"),
    ("votants", "votants"),
    ("exprimes", "exprimes"),
]
# Output columns. Blank and null ballots are only available together in 2002 ("Blancs ou nuls"),
# so only their total is kept; in 2022 "Blancs" and "Nuls" are parsed and summed.
TURNOUT_COLUMNS = ["inscrits", "abstentions", "votants", "blancs_nuls", "exprimes"]

ROUND_RE = re.compile(r"\b(1er|1 er|premier|2nd|2 nd|2d|2e|2eme|second|deuxieme)\s+tour\b")
VOTES_HEADER_RE = r"^(voix|nombre de voix)$"


def table_rows(table):
    """List of rows, each a list of cell texts (inner spacing kept, e.g. 'M.  JACQUES  CHIRAC')."""
    rows = []
    for tr in table.find_all("tr"):
        cells = [c.get_text(" ", strip=True) for c in tr.find_all(["th", "td"])]
        if any(cells):
            rows.append(cells)
    return rows


def turnout_label(cell):
    key = re.sub(r"^(nombre (de |d )?|bulletins |votes )", "", norm_text(cell))
    for prefix, field in TURNOUT_FIELDS:
        if key.startswith(prefix):
            return field
    return None


def header_index(rows, pattern):
    for i, row in enumerate(rows):
        if any(re.search(pattern, norm_text(c)) for c in row):
            return i
    return None


def split_surname(raw_name):
    """2002 pages separate civility, first name and surname with double spaces:
    'M.  JEAN-MARIE  LE PEN' -> 'LE PEN'. None if the name is not written that way."""
    parts = re.split(r"\s{2,}", raw_name.strip())
    return parts[-1] if len(parts) >= 3 else None


def parse_candidates(rows):
    """[{'candidate', 'candidate_surname', 'votes', 'vote_share'}] from a table with a 'Voix' column; [] otherwise."""
    h = header_index(rows, VOTES_HEADER_RE)
    if h is None:
        return []
    header = [norm_text(c) for c in rows[h]]
    votes_col = next(i for i, c in enumerate(header) if re.search(VOTES_HEADER_RE, c))
    share_col = next((i for i, c in enumerate(header) if "exprim" in c), None)
    name_col = next((i for i, c in enumerate(header) if re.search(r"^(candidats?|listes?|noms?)\b", c)), 0)

    candidates = []
    for row in rows[h + 1:]:
        if len(row) != len(header) or turnout_label(row[0]) or turnout_label(row[name_col]):
            continue
        votes = to_int(row[votes_col])
        if votes is None or not re.search(r"[A-Za-z]", row[name_col]):
            continue
        candidates.append({
            "candidate": " ".join(row[name_col].split()),
            "candidate_surname": split_surname(row[name_col]),
            "votes": votes,
            "vote_share": to_float(row[share_col]) if share_col is not None else None,
        })
    return candidates


def parse_turnout(rows):
    """Turnout figures found in a table, e.g. {'inscrits': ..., 'votants': ...}.

    May be partial: on the 2002 pages Inscrits/Abstentions/Votants and Blancs ou nuls/Exprimés
    are two separate tables. None if the table contains no turnout line.
    """
    h = header_index(rows, r"^nombre$")
    col = [norm_text(c) for c in rows[h]].index("nombre") if h is not None else 1

    values = {}
    for row in rows:
        field = turnout_label(row[0])
        if field and field not in values and len(row) > col:
            values[field] = to_int(row[col])

    # Transposed layout: labels in one row, numbers in the next one
    if not values:
        for i, row in enumerate(rows[:-1]):
            fields = [turnout_label(c) for c in row]
            if sum(f is not None for f in fields) >= 3:
                values = {f: to_int(v) for f, v in zip(fields, rows[i + 1]) if f}
                break

    values = {k: v for k, v in values.items() if v is not None}
    return values or None


def guess_round(table):
    """1 or 2 if the table's caption, container id/class or nearest preceding text says so, else None."""
    texts = [table.caption.get_text(" ")] if table.caption else []
    for parent in [table, *table.parents]:
        attrs = " ".join([parent.get("id") or "", *(parent.get("class") or [])]) if hasattr(parent, "get") else ""
        m = re.search(r"tour\s?([12])\b|\bt([12])\b", norm_text(attrs))
        if m:
            return int(m.group(1) or m.group(2))
    texts += list(table.find_all_previous(string=True, limit=40))
    for text in texts:
        m = ROUND_RE.search(norm_text(text))
        if m:
            return 1 if m.group(1) in ("1er", "1 er", "premier") else 2
    return None


def extract_first_round(soup, year):
    """Return (turnout, candidates, info) for the first round shown on a page."""
    cand_tables, turnout_tables = [], []
    for pos, table in enumerate(soup.find_all("table")):
        if table.find("table"):  # layout table wrapping other tables
            continue
        rows = table_rows(table)
        rnd = guess_round(table)
        cands, turnout = parse_candidates(rows), parse_turnout(rows)
        if cands:
            cand_tables.append({"pos": pos, "round": rnd, "cands": cands})
        if turnout:
            turnout_tables.append({"pos": pos, "round": rnd, "values": turnout})

    first_round = [t for t in cand_tables if len(t["cands"]) > 2]  # round 2 has exactly 2 candidates
    if not first_round:
        raise ValueError(f"no first-round candidate table (candidate tables sizes: {[len(t['cands']) for t in cand_tables]})")
    first_round.sort(key=lambda t: (t["round"] == 2, len(t["cands"]) != N_CANDIDATES[year], t["pos"]))
    cand = first_round[0]

    # Turnout tables belonging to this candidate table are either printed just above it
    # (between the previous candidate table and this one: 2002 layout) or just below it
    # (between this one and the next candidate table: 2022 layout). Both blocks exist on a page
    # showing two rounds, and one of them belongs to the other round, so keep the block whose
    # 'Exprimés' equals the sum of the candidates' votes.
    total = sum(c["votes"] for c in cand["cands"])
    positions = [t["pos"] for t in cand_tables]
    prev_pos = max((p for p in positions if p < cand["pos"]), default=-1)
    next_pos = min((p for p in positions if p > cand["pos"]), default=float("inf"))
    blocks = [
        [t for t in turnout_tables if prev_pos < t["pos"] <= cand["pos"]],  # above
        [t for t in turnout_tables if cand["pos"] <= t["pos"] < next_pos],  # below
    ]
    merged = []
    for block in blocks:
        turnout = {}
        for t in block:
            for field, value in t["values"].items():
                turnout.setdefault(field, value)
        if "exprimes" in turnout and "inscrits" in turnout:
            merged.append(turnout)
    if not merged:
        raise ValueError("no turnout (Inscrits ... Exprimés) found next to the first-round candidates; "
                         f"turnout fields per table on the page: {[sorted(t['values']) for t in turnout_tables]}")
    # Prefer the block consistent with the candidate votes; otherwise keep the first one
    # (flagged by turnout_matches_sum = False and caught by the validation checks)
    turnout = next((t for t in merged if t["exprimes"] == total), merged[0])
    info = {
        "table_round_label": cand["round"],
        "turnout_matches_sum": turnout["exprimes"] == total,
        "page_heading": page_heading(soup),
    }
    return turnout, cand["cands"], info


def parse_results_page(url, year):
    """Fetch a page and extract round 1; follow a '1er tour' link if round 1 is not on the page."""
    soup = get_soup(url)
    try:
        return extract_first_round(soup, year) + (url,)
    except ValueError:
        for link, label in iter_links(soup, url):
            m = ROUND_RE.search(norm_text(label))
            if m and m.group(1) in ("1er", "1 er", "premier") and link != url:
                return extract_first_round(get_soup(link), year) + (link,)
        raise


def build_frame(year, dep_code, dep_name, turnout, candidates, info, url):
    """Long format: one row per candidate, turnout figures repeated on each row."""
    base = {"year": year, "round": 1, "dep_code": dep_code, "dep_name": dep_name}
    base.update({c: turnout.get(c) for c in TURNOUT_COLUMNS})
    if base["blancs_nuls"] is None and "blancs" in turnout and "nuls" in turnout:  # 2022: counted separately
        base["blancs_nuls"] = turnout["blancs"] + turnout["nuls"]
    df = pd.DataFrame([{**base, **c} for c in candidates])
    df["source_url"] = url
    for key, value in info.items():
        df[key] = value
    return df


def describe_tables(url, n_rows=5):
    """Debugging: print a summary of every table on a page."""
    soup = get_soup(url)
    print("Heading:", page_heading(soup))
    for pos, table in enumerate(soup.find_all("table")):
        rows = table_rows(table)
        print(f"\n--- table {pos}: {len(rows)} rows | round label: {guess_round(table)} | "
              f"layout table: {bool(table.find('table'))}")
        for row in rows[:n_rows]:
            print("   ", row)

### Scraping loop (shared by both years)

In [109]:
def scrape_year(year, dep_urls, parser, limit=SCRAPE_LIMIT):
    """Parse every département page; failures are recorded (not dropped) and the loop continues."""
    todo = dep_urls if limit is None else dep_urls.head(limit)
    frames, failures = [], []
    for i, row in enumerate(todo.itertuples(index=False), 1):
        try:
            frames.append(parser(row.url, row.dep_code, row.dep_name))
        except Exception as e:
            failures.append({"year": year, "dep_code": row.dep_code, "dep_name": row.dep_name,
                             "url": row.url, "error": repr(e)})
            print(f"  FAILED {row.dep_code} {row.dep_name}: {e!r}")
        if i % 10 == 0:
            print(f"  {i}/{len(todo)} pages done")

    results = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    failures = pd.DataFrame(failures, columns=["year", "dep_code", "dep_name", "url", "error"])
    print(f"{year}: {len(frames)} départements parsed, {len(failures)} failures")
    return results, failures


def retry_failures(year, dep_urls, results, failures, parser):
    """Re-run the failed départements only and merge them into the results."""
    todo = dep_urls[dep_urls["dep_code"].isin(failures["dep_code"])]
    new_results, new_failures = scrape_year(year, todo, parser, limit=None)
    results = pd.concat([results, new_results], ignore_index=True)
    return results, new_failures

## 4. 2002 — discover département URLs

If this finds few or no départements, inspect the start page:
`show_links(START_URLS[2002])` (all links) or `show_links(START_URLS[2002], "ain|paris")`.

In [110]:
dep_urls_2002 = discover_departement_urls(2002)
dep_urls_2002.to_csv(RAW_DIR / "presidential_2002_department_urls.csv", index=False, encoding="utf-8")
dep_urls_2002.head(10)

2002: visited 1 pages, 249 département links found
2002: 105 distinct départements/territories; missing (strict sense): none


,year,dep_code,dep_name,dep_type,url,link_label,method,found_on
0,2002,01,Ain,metropole,https://www.archives-resultats-elections.inter...,01- AIN,link label,https://www.archives-resultats-elections.inter...
1,2002,02,Aisne,metropole,https://www.archives-resultats-elections.inter...,02- AISNE,link label,https://www.archives-resultats-elections.inter...
2,2002,03,Allier,metropole,https://www.archives-resultats-elections.inter...,03- ALLIER,link label,https://www.archives-resultats-elections.inter...
3,2002,04,Alpes-de-Haute-Provence,metropole,https://www.archives-resultats-elections.inter...,04- ALPES DE HAUTE PROVENCE,link label,https://www.archives-resultats-elections.inter...
4,2002,05,Hautes-Alpes,metropole,https://www.archives-resultats-elections.inter...,05- HAUTES ALPES,link label,https://www.archives-resultats-elections.inter...
5,2002,06,Alpes-Maritimes,metropole,https://www.archives-resultats-elections.inter...,06- ALPES MARITIMES,link label,https://www.archives-resultats-elections.inter...
6,2002,07,Ardèche,metropole,https://www.archives-resultats-elections.inter...,07- ARDECHE,link label,https://www.archives-resultats-elections.inter...
7,2002,08,Ardennes,metropole,https://www.archives-resultats-elections.inter...,08- ARDENNES,link label,https://www.archives-resultats-elections.inter...
8,2002,09,Ariège,metropole,https://www.archives-resultats-elections.inter...,09- ARIEGE,link label,https://www.archives-resultats-elections.inter...
9,2002,10,Aube,metropole,https://www.archives-resultats-elections.inter...,10- AUBE,link label,https://www.archives-resultats-elections.inter...


In [111]:
# Debugging (uncomment if needed)
# show_links(START_URLS[2002])
dep_urls_2002["method"].value_counts()

method
link label    105
Name: count, dtype: int64

## 5. 2002 — parse one département page

2002 pages are expected to report **`Blancs ou nuls`** as a single line (blank and null ballots were
not counted separately before 2014) and 16 candidates. `describe_tables` shows what the parser sees;
adapt `parse_departement_2002` if the structure differs.

In [112]:
def parse_departement_2002(url, dep_code, dep_name):
    turnout, candidates, info, page_url = parse_results_page(url, 2002)
    return build_frame(2002, dep_code, dep_name, turnout, candidates, info, page_url)


test_2002 = dep_urls_2002.set_index("dep_code").loc["75"]  # Paris
describe_tables(test_2002["url"])

Heading: PARIS (75) (résultats officiels)

--- table 0: 38 rows | round label: None | layout table: True
    ['', "RESULTATS DE L'ELECTION PRESIDENTIELLE"]
    ['', 'DIMANCHE 5 MAI 2002 PARIS\n\n\t(75) (résultats officiels)']
    ['', 'Résultats par arrondissement : Cliquez sur le n° de votre arrondissement 1er 2ème 3ème 4ème 5ème 6ème 7ème 8ème 9ème 10ème 11ème 12ème 13ème 14ème 15ème 16ème 17ème 18ème 19ème 20ème']
    ['Nombre % Inscrits Inscrits 1 081 313 100,00 Abstentions 184 502 17,06 Votants 896 811 82,94 Nombre % Votants Blancs ou nuls 24 569 2,74 Exprimés 872 242 97,26 Voix % Exprimés M.\xa0\xa0JACQUES\xa0\xa0CHIRAC 784 741 89,97 M.\xa0\xa0JEAN-MARIE\xa0\xa0LE PEN 87 501 10,03 RAPPEL DES RESULTATS 1er tour DIMANCHE 21 AVRIL 2002 Nombre % Inscrits Inscrits 1 081 420 100,00 Abstentions 322 390 29,81 Votants 759 030 70,19 Nombre % Votants Blancs ou Nuls 14 970 1,97 Exprimés 744 060 98,03 Voix % Exprimés M.\xa0\xa0BRUNO\xa0\xa0MEGRET 7 609 1,02 Mme\xa0\xa0CORINNE\xa0\xa0LEPAGE 18

In [113]:
parse_departement_2002(test_2002["url"], "75", test_2002["dep_name"])

,year,round,dep_code,dep_name,inscrits,abstentions,votants,blancs_nuls,exprimes,candidate,candidate_surname,votes,vote_share,source_url,table_round_label,turnout_matches_sum,page_heading
0,2002,1,75,Paris,1081420,322390,759030,14970,744060,M. BRUNO MEGRET,MEGRET,7609,1.02,https://www.archives-resultats-elections.inter...,None,True,PARIS (75) (résultats officiels)
1,2002,1,75,Paris,1081420,322390,759030,14970,744060,Mme CORINNE LEPAGE,LEPAGE,18272,2.46,https://www.archives-resultats-elections.inter...,None,True,PARIS (75) (résultats officiels)
2,2002,1,75,Paris,1081420,322390,759030,14970,744060,M. DANIEL GLUCKSTEIN,GLUCKSTEIN,2494,0.34,https://www.archives-resultats-elections.inter...,None,True,PARIS (75) (résultats officiels)
3,2002,1,75,Paris,1081420,322390,759030,14970,744060,M. FRANCOIS BAYROU,BAYROU,58869,7.91,https://www.archives-resultats-elections.inter...,None,True,PARIS (75) (résultats officiels)
4,2002,1,75,Paris,1081420,322390,759030,14970,744060,M. JACQUES CHIRAC,CHIRAC,178658,24.01,https://www.archives-resultats-elections.inter...,None,True,PARIS (75) (résultats officiels)
5,2002,1,75,Paris,1081420,322390,759030,14970,744060,M. JEAN-MARIE LE PEN,LE PEN,69583,9.35,https://www.archives-resultats-elections.inter...,None,True,PARIS (75) (résultats officiels)
6,2002,1,75,Paris,1081420,322390,759030,14970,744060,Mme CHRISTIANE TAUBIRA,TAUBIRA,28157,3.78,https://www.archives-resultats-elections.inter...,None,True,PARIS (75) (résultats officiels)
7,2002,1,75,Paris,1081420,322390,759030,14970,744060,M. JEAN SAINT-JOSSE,SAINT-JOSSE,3899,0.52,https://www.archives-resultats-elections.inter...,None,True,PARIS (75) (résultats officiels)
8,2002,1,75,Paris,1081420,322390,759030,14970,744060,M. NOEL MAMERE,MAMERE,54964,7.39,https://www.archives-resultats-elections.inter...,None,True,PARIS (75) (résultats officiels)
9,2002,1,75,Paris,1081420,322390,759030,14970,744060,M. LIONEL JOSPIN,JOSPIN,148401,19.94,https://www.archives-resultats-elections.inter...,None,True,PARIS (75) (résultats officiels)


## 6. 2002 — scraping loop

In [114]:
results_2002, failures_2002 = scrape_year(2002, dep_urls_2002, parse_departement_2002)
failures_2002

  10/105 pages done
  20/105 pages done
  30/105 pages done
  40/105 pages done
  50/105 pages done
  60/105 pages done
  70/105 pages done
  80/105 pages done
  90/105 pages done
  100/105 pages done
2002: 105 départements parsed, 0 failures


,year,dep_code,dep_name,url,error


In [115]:
# If some pages failed (e.g. timeout), retry only those:
# results_2002, failures_2002 = retry_failures(2002, dep_urls_2002, results_2002, failures_2002, parse_departement_2002)
results_2002.head()

,year,round,dep_code,dep_name,inscrits,abstentions,votants,blancs_nuls,exprimes,candidate,candidate_surname,votes,vote_share,source_url,table_round_label,turnout_matches_sum,page_heading
0,2002,1,01,Ain,338220,89002,249218,8566,240652,M. BRUNO MEGRET,MEGRET,8425,3.50,https://www.archives-resultats-elections.inter...,None,True,AIN (01) (résultats officiels)
1,2002,1,01,Ain,338220,89002,249218,8566,240652,Mme CORINNE LEPAGE,LEPAGE,5258,2.18,https://www.archives-resultats-elections.inter...,None,True,AIN (01) (résultats officiels)
2,2002,1,01,Ain,338220,89002,249218,8566,240652,M. DANIEL GLUCKSTEIN,GLUCKSTEIN,1077,0.45,https://www.archives-resultats-elections.inter...,None,True,AIN (01) (résultats officiels)
3,2002,1,01,Ain,338220,89002,249218,8566,240652,M. FRANCOIS BAYROU,BAYROU,18614,7.73,https://www.archives-resultats-elections.inter...,None,True,AIN (01) (résultats officiels)
4,2002,1,01,Ain,338220,89002,249218,8566,240652,M. JACQUES CHIRAC,CHIRAC,41348,17.18,https://www.archives-resultats-elections.inter...,None,True,AIN (01) (résultats officiels)


## 7. 2022 — discover département URLs

The 2022 site may organise départements under their region (e.g. `.../presidentielle-2022/084/001/index.php`
for Ain in Auvergne-Rhône-Alpes); the crawler follows region pages automatically (`max_depth=2`).

In [116]:
dep_urls_2022 = discover_departement_urls(2022)
dep_urls_2022.to_csv(RAW_DIR / "presidential_2022_department_urls.csv", index=False, encoding="utf-8")
dep_urls_2022.head(10)

2022: visited 1 pages, 470 département links found
2022: 108 distinct départements/territories; missing (strict sense): none


,year,dep_code,dep_name,dep_type,url,link_label,method,found_on
0,2022,01,Ain,metropole,https://www.archives-resultats-elections.inter...,01 - Ain,link label,https://www.archives-resultats-elections.inter...
1,2022,02,Aisne,metropole,https://www.archives-resultats-elections.inter...,02 - Aisne,link label,https://www.archives-resultats-elections.inter...
2,2022,03,Allier,metropole,https://www.archives-resultats-elections.inter...,03 - Allier,link label,https://www.archives-resultats-elections.inter...
3,2022,04,Alpes-de-Haute-Provence,metropole,https://www.archives-resultats-elections.inter...,04 - Alpes-de-Haute-Provence,link label,https://www.archives-resultats-elections.inter...
4,2022,05,Hautes-Alpes,metropole,https://www.archives-resultats-elections.inter...,05 - Hautes-Alpes,link label,https://www.archives-resultats-elections.inter...
5,2022,06,Alpes-Maritimes,metropole,https://www.archives-resultats-elections.inter...,06 - Alpes-Maritimes,link label,https://www.archives-resultats-elections.inter...
6,2022,07,Ardèche,metropole,https://www.archives-resultats-elections.inter...,07 - Ardèche,link label,https://www.archives-resultats-elections.inter...
7,2022,08,Ardennes,metropole,https://www.archives-resultats-elections.inter...,08 - Ardennes,link label,https://www.archives-resultats-elections.inter...
8,2022,09,Ariège,metropole,https://www.archives-resultats-elections.inter...,09 - Ariège,link label,https://www.archives-resultats-elections.inter...
9,2022,10,Aube,metropole,https://www.archives-resultats-elections.inter...,10 - Aube,link label,https://www.archives-resultats-elections.inter...


In [117]:
# Debugging (uncomment if needed)
# show_links(START_URLS[2022])
dep_urls_2022["method"].value_counts()

method
link label     107
url pattern      1
Name: count, dtype: int64

## 8. 2022 — parse one département page

2022 pages are expected to report **`Blancs`** and **`Nuls`** on separate lines and 12 candidates.
Only their sum is kept (`blancs_nuls`), to match 2002 where they are not reported separately.

In [118]:
def parse_departement_2022(url, dep_code, dep_name):
    turnout, candidates, info, page_url = parse_results_page(url, 2022)
    return build_frame(2022, dep_code, dep_name, turnout, candidates, info, page_url)


test_2022 = dep_urls_2022.set_index("dep_code").loc["75"]  # Paris
describe_tables(test_2022["url"])

Heading: Les archives des élections en France

--- table 0: 3 rows | round label: None | layout table: False
    ['Liste des candidats', 'Voix', '% Inscrits', '% Exprimés']
    ['M. Emmanuel MACRON', '808 107', '59,05', '85,11']
    ['Mme Marine LE PEN', '141 405', '10,33', '14,89']

--- table 1: 7 rows | round label: None | layout table: False
    ['', 'Nombre', '% Inscrits', '% Votants']
    ['Inscrits', '1 368 623', '', '']
    ['Abstentions', '354 494', '25,90', '']
    ['Votants', '1 014 129', '74,10', '']
    ['Blancs', '49 446', '3,61', '4,88']

--- table 2: 13 rows | round label: None | layout table: False
    ['Liste des candidats', 'Voix', '% Inscrits', '% Exprimés']
    ['M. Emmanuel MACRON', '372 820', '27,25', '35,34']
    ['M. Jean-Luc MÉLENCHON', '317 372', '23,20', '30,08']
    ['M. Éric ZEMMOUR', '86 088', '6,29', '8,16']
    ['M. Yannick JADOT', '80 268', '5,87', '7,61']

--- table 3: 7 rows | round label: None | layout table: False
    ['', 'Nombre', '% Inscrits', '%

In [119]:
parse_departement_2022(test_2022["url"], "75", test_2022["dep_name"])

,year,round,dep_code,dep_name,inscrits,abstentions,votants,blancs_nuls,exprimes,candidate,candidate_surname,votes,vote_share,source_url,table_round_label,turnout_matches_sum,page_heading
0,2022,1,75,Paris,1368025,296668,1071357,16295,1055062,M. Emmanuel MACRON,None,372820,35.34,https://www.archives-resultats-elections.inter...,None,True,Les archives des élections en France
1,2022,1,75,Paris,1368025,296668,1071357,16295,1055062,M. Jean-Luc MÉLENCHON,None,317372,30.08,https://www.archives-resultats-elections.inter...,None,True,Les archives des élections en France
2,2022,1,75,Paris,1368025,296668,1071357,16295,1055062,M. Éric ZEMMOUR,None,86088,8.16,https://www.archives-resultats-elections.inter...,None,True,Les archives des élections en France
3,2022,1,75,Paris,1368025,296668,1071357,16295,1055062,M. Yannick JADOT,None,80268,7.61,https://www.archives-resultats-elections.inter...,None,True,Les archives des élections en France
4,2022,1,75,Paris,1368025,296668,1071357,16295,1055062,Mme Valérie PÉCRESSE,None,69564,6.59,https://www.archives-resultats-elections.inter...,None,True,Les archives des élections en France
5,2022,1,75,Paris,1368025,296668,1071357,16295,1055062,Mme Marine LE PEN,None,58429,5.54,https://www.archives-resultats-elections.inter...,None,True,Les archives des élections en France
6,2022,1,75,Paris,1368025,296668,1071357,16295,1055062,Mme Anne HIDALGO,None,22901,2.17,https://www.archives-resultats-elections.inter...,None,True,Les archives des élections en France
7,2022,1,75,Paris,1368025,296668,1071357,16295,1055062,M. Fabien ROUSSEL,None,17267,1.64,https://www.archives-resultats-elections.inter...,None,True,Les archives des élections en France
8,2022,1,75,Paris,1368025,296668,1071357,16295,1055062,M. Jean LASSALLE,None,12139,1.15,https://www.archives-resultats-elections.inter...,None,True,Les archives des élections en France
9,2022,1,75,Paris,1368025,296668,1071357,16295,1055062,M. Nicolas DUPONT-AIGNAN,None,9591,0.91,https://www.archives-resultats-elections.inter...,None,True,Les archives des élections en France


## 9. 2022 — scraping loop

In [120]:
results_2022, failures_2022 = scrape_year(2022, dep_urls_2022, parse_departement_2022)
failures_2022

  10/108 pages done
  20/108 pages done
  30/108 pages done
  40/108 pages done
  50/108 pages done
  60/108 pages done
  70/108 pages done
  80/108 pages done
  90/108 pages done
  100/108 pages done
2022: 108 départements parsed, 0 failures


,year,dep_code,dep_name,url,error


In [121]:
# If some pages failed (e.g. timeout), retry only those:
# results_2022, failures_2022 = retry_failures(2022, dep_urls_2022, results_2022, failures_2022, parse_departement_2022)
results_2022.head()

,year,round,dep_code,dep_name,inscrits,abstentions,votants,blancs_nuls,exprimes,candidate,candidate_surname,votes,vote_share,source_url,table_round_label,turnout_matches_sum,page_heading
0,2022,1,01,Ain,438109,97541,340568,7544,333024,M. Emmanuel MACRON,None,92206,27.69,https://www.archives-resultats-elections.inter...,None,True,Les archives des élections en France
1,2022,1,01,Ain,438109,97541,340568,7544,333024,Mme Marine LE PEN,None,86755,26.05,https://www.archives-resultats-elections.inter...,None,True,Les archives des élections en France
2,2022,1,01,Ain,438109,97541,340568,7544,333024,M. Jean-Luc MÉLENCHON,None,57832,17.37,https://www.archives-resultats-elections.inter...,None,True,Les archives des élections en France
3,2022,1,01,Ain,438109,97541,340568,7544,333024,M. Éric ZEMMOUR,None,27530,8.27,https://www.archives-resultats-elections.inter...,None,True,Les archives des élections en France
4,2022,1,01,Ain,438109,97541,340568,7544,333024,Mme Valérie PÉCRESSE,None,17572,5.28,https://www.archives-resultats-elections.inter...,None,True,Les archives des élections en France


## 10. Combine datasets

Adds `dep_type` (metropole / DOM / COM / etranger), a cleaned candidate name without civility
(`M.`, `Mme`), the candidate surname (upper-case part of the name, e.g. `LE PEN`) for matching across years,
and a vote share recomputed from the counts.

In [122]:
CIVILITY_RE = r"^(M\.|Mme\.?|Mlle\.?|Monsieur|Madame)\s+"


def surname(name):
    """Upper-case words of the name: 'Mme Marine LE PEN' -> 'LE PEN'."""
    words = re.findall(r"[^\s]+", re.sub(CIVILITY_RE, "", name))
    upper = [w for w in words if len(w) > 1 and w.upper() == w and re.search(r"[A-Z]", strip_accents(w))]
    return " ".join(upper) if upper else name


results = pd.concat([results_2002, results_2022], ignore_index=True)
results = results.merge(DEPARTEMENTS[["dep_code", "dep_type"]], on="dep_code", how="left")
results["candidate_clean"] = results["candidate"].str.replace(CIVILITY_RE, "", regex=True).str.strip()
# 2002 names are written "M.  JEAN-MARIE  LE PEN": the surname was split at parse time;
# otherwise fall back to the upper-case words of the name ("Mme Marine LE PEN" -> "LE PEN")
results["candidate_surname"] = results["candidate_surname"].fillna(results["candidate"].map(surname))
results["vote_share_calc"] = 100 * results["votes"] / results["exprimes"]

results = results[[
    "year", "round", "dep_code", "dep_name", "dep_type",
    "inscrits", "abstentions", "votants", "blancs_nuls", "exprimes",
    "candidate", "candidate_clean", "candidate_surname", "votes", "vote_share", "vote_share_calc",
    "source_url", "page_heading", "table_round_label", "turnout_matches_sum",
]].sort_values(["year", "dep_code", "votes"], ascending=[True, True, False]).reset_index(drop=True)

results.head()

,year,round,dep_code,dep_name,dep_type,inscrits,abstentions,votants,blancs_nuls,exprimes,candidate,candidate_clean,candidate_surname,votes,vote_share,vote_share_calc,source_url,page_heading,table_round_label,turnout_matches_sum
0,2002,1,01,Ain,metropole,338220,89002,249218,8566,240652,M. JEAN-MARIE LE PEN,JEAN-MARIE LE PEN,LE PEN,52617,21.86,21.864352,https://www.archives-resultats-elections.inter...,AIN (01) (résultats officiels),None,True
1,2002,1,01,Ain,metropole,338220,89002,249218,8566,240652,M. JACQUES CHIRAC,JACQUES CHIRAC,CHIRAC,41348,17.18,17.181656,https://www.archives-resultats-elections.inter...,AIN (01) (résultats officiels),None,True
2,2002,1,01,Ain,metropole,338220,89002,249218,8566,240652,M. LIONEL JOSPIN,LIONEL JOSPIN,JOSPIN,30418,12.64,12.639828,https://www.archives-resultats-elections.inter...,AIN (01) (résultats officiels),None,True
3,2002,1,01,Ain,metropole,338220,89002,249218,8566,240652,M. FRANCOIS BAYROU,FRANCOIS BAYROU,BAYROU,18614,7.73,7.734820,https://www.archives-resultats-elections.inter...,AIN (01) (résultats officiels),None,True
4,2002,1,01,Ain,metropole,338220,89002,249218,8566,240652,M. JEAN-PIERRE CHEVENEMENT,JEAN-PIERRE CHEVENEMENT,CHEVENEMENT,14665,6.09,6.093862,https://www.archives-resultats-elections.inter...,AIN (01) (résultats officiels),None,True


## 11. Validation checks

In [123]:
def report(label, ok, detail=None):
    print(f"[{'OK' if ok else 'CHECK'}] {label}")
    if not ok and detail is not None and len(detail):
        display(detail)


# One row per year × département (turnout figures)
deps = results.drop_duplicates(["year", "dep_code"])

### Number of départements, unique codes, missing départements

In [124]:
for year in (2002, 2022):
    codes = deps.loc[deps["year"] == year, "dep_code"]
    print(f"\n{year}: {codes.nunique()} départements/territories with results "
          f"({len(EXPECTED_DEPS[year])} départements expected in the strict sense)")
    report(f"{year}: département codes are unique", not codes.duplicated().any(), codes[codes.duplicated()])
    missing = sorted(EXPECTED_DEPS[year] - set(codes))
    report(f"{year}: no missing département — missing: {missing}", not missing)
    extras = sorted(set(codes) - EXPECTED_DEPS[year])
    print(f"   other territories included: {extras}")

report("no duplicated year × département × candidate rows",
       not results.duplicated(["year", "dep_code", "candidate"]).any(),
       results[results.duplicated(["year", "dep_code", "candidate"], keep=False)])

report("no scraping failures",
       failures_2002.empty and failures_2022.empty,
       pd.concat([failures_2002, failures_2022]))


2002: 105 départements/territories with results (100 départements expected in the strict sense)
[OK] 2002: département codes are unique
[OK] 2002: no missing département — missing: []
   other territories included: ['975', '976', '986', '987', '988']

2022: 108 départements/territories with results (101 départements expected in the strict sense)
[OK] 2022: département codes are unique
[OK] 2022: no missing département — missing: []
   other territories included: ['975', '977', '986', '987', '988', 'ZX', 'ZZ']
[OK] no duplicated year × département × candidate rows
[OK] no scraping failures


### Number of candidates per election

In [125]:
for year in (2002, 2022):
    sub = results[results["year"] == year]
    n_by_dep = sub.groupby("dep_code")["candidate"].nunique()
    print(f"\n{year}: {sub['candidate_clean'].nunique()} distinct candidates overall (expected {N_CANDIDATES[year]})")
    report(f"{year}: every département has {N_CANDIDATES[year]} candidates",
           (n_by_dep == N_CANDIDATES[year]).all(), n_by_dep[n_by_dep != N_CANDIDATES[year]])
    display(sub.groupby("candidate_clean")["votes"].sum().sort_values(ascending=False).to_frame("national_votes"))


2002: 16 distinct candidates overall (expected 16)
[OK] 2002: every département has 16 candidates


,national_votes
candidate_clean,
JACQUES CHIRAC,5622601
JEAN-MARIE LE PEN,4795541
LIONEL JOSPIN,4577921
FRANCOIS BAYROU,1940585
ARLETTE LAGUILLER,1626112
JEAN-PIERRE CHEVENEMENT,1510436
NOEL MAMERE,1485615
OLIVIER BESANCENOT,1207732
JEAN SAINT-JOSSE,1203909



2022: 12 distinct candidates overall (expected 12)
[OK] 2022: every département has 12 candidates


,national_votes
candidate_clean,
Emmanuel MACRON,9785128
Marine LE PEN,8135273
Jean-Luc MÉLENCHON,7714874
Éric ZEMMOUR,2486333
Valérie PÉCRESSE,1679355
Yannick JADOT,1628100
Jean LASSALLE,1101595
Fabien ROUSSEL,802470
Nicolas DUPONT-AIGNAN,725515


### Candidate votes sum to `Exprimés`; shares sum to ~100 %; turnout accounting

In [126]:
by_dep = results.groupby(["year", "dep_code"]).agg(
    exprimes=("exprimes", "first"),
    sum_votes=("votes", "sum"),
    sum_share=("vote_share", "sum"),
    sum_share_calc=("vote_share_calc", "sum"),
).reset_index()
by_dep["diff_votes"] = by_dep["sum_votes"] - by_dep["exprimes"]

report("sum of candidate votes == Exprimés in every département",
       (by_dep["diff_votes"] == 0).all(), by_dep[by_dep["diff_votes"] != 0])

# Published shares are rounded to 2 decimals: allow a small tolerance
bad_share = by_dep[(by_dep["sum_share"] - 100).abs() > 0.2]
report("published vote shares sum to ~100 % (±0.2)", bad_share.empty, bad_share)
report("recomputed vote shares sum to 100 %", ((by_dep["sum_share_calc"] - 100).abs() < 1e-6).all())

share_gap = (results["vote_share"] - results["vote_share_calc"]).abs()
report("published shares match recomputed shares (±0.01)", (share_gap <= 0.011).all(),
       results.loc[share_gap > 0.011, ["year", "dep_code", "candidate", "votes", "exprimes", "vote_share", "vote_share_calc"]])

acc = deps.copy()
acc["votants_check"] = acc["inscrits"] - acc["abstentions"]
acc["votants_check2"] = acc["exprimes"] + acc["blancs_nuls"]
report("Votants == Inscrits − Abstentions", (acc["votants"] == acc["votants_check"]).all(),
       acc.loc[acc["votants"] != acc["votants_check"], ["year", "dep_code", "inscrits", "abstentions", "votants"]])
report("Votants == Exprimés + Blancs/nuls", (acc["votants"] == acc["votants_check2"]).all(),
       acc.loc[acc["votants"] != acc["votants_check2"], ["year", "dep_code", "votants", "exprimes", "blancs_nuls"]])
report("no missing turnout values", acc[["inscrits", "abstentions", "votants", "blancs_nuls", "exprimes"]].notna().all().all(),
       acc[acc[["inscrits", "abstentions", "votants", "blancs_nuls", "exprimes"]].isna().any(axis=1)])

[OK] sum of candidate votes == Exprimés in every département
[OK] published vote shares sum to ~100 % (±0.2)
[OK] recomputed vote shares sum to 100 %
[OK] published shares match recomputed shares (±0.01)
[OK] Votants == Inscrits − Abstentions
[OK] Votants == Exprimés + Blancs/nuls
[OK] no missing turnout values


### No mixing of first and second rounds

* no département with only 2 candidates;
* candidates who ran only in round 1 (e.g. Jospin 2002, Zemmour 2022) appear in every département;
* the set of candidates is identical in all départements of a given year;
* no table explicitly labelled `2nd tour` was used.

In [127]:
report("round column is always 1", results["round"].eq(1).all())

n_by_dep = results.groupby(["year", "dep_code"])["candidate"].nunique()
report("no département with only 2 candidates (round 2)", (n_by_dep > 2).all(), n_by_dep[n_by_dep <= 2])

for year, names in FIRST_ROUND_ONLY.items():
    sub = results[results["year"] == year]
    # words of all candidate names in each département, upper-case without accents
    words = (sub.assign(w=sub["candidate"].map(lambda c: set(norm_text(c).upper().split())))
                .groupby("dep_code")["w"].apply(lambda ws: set().union(*ws)))
    for name in names:
        lacking = words[~words.map(lambda s: name in s)]
        report(f"{year}: first-round-only candidate {name} present in every département", lacking.empty, lacking)

    candidate_sets = sub.groupby("dep_code")["candidate_clean"].apply(frozenset)
    report(f"{year}: same candidate list in every département", candidate_sets.nunique() == 1,
           candidate_sets.value_counts().to_frame("n_departements"))

report("no table labelled '2nd tour' used", not results["table_round_label"].eq(2).any(),
       deps.loc[deps["table_round_label"].eq(2), ["year", "dep_code", "source_url"]])
report("turnout table matches candidate table in every département", deps["turnout_matches_sum"].all(),
       deps.loc[~deps["turnout_matches_sum"], ["year", "dep_code", "source_url"]])
print("table_round_label counts (None = no label found near the table):")
print(deps["table_round_label"].value_counts(dropna=False))

[OK] round column is always 1
[OK] no département with only 2 candidates (round 2)
[OK] 2002: first-round-only candidate JOSPIN present in every département
[OK] 2002: first-round-only candidate BAYROU present in every département
[OK] 2002: same candidate list in every département
[OK] 2022: first-round-only candidate ZEMMOUR present in every département
[OK] 2022: first-round-only candidate PECRESSE present in every département
[OK] 2022: same candidate list in every département
[OK] no table labelled '2nd tour' used
[OK] turnout table matches candidate table in every département
table_round_label counts (None = no label found near the table):
table_round_label
None    213
Name: count, dtype: int64


### Corsica (`2A`, `2B`) and overseas départements

In [128]:
for year in (2002, 2022):
    codes = set(deps.loc[deps["year"] == year, "dep_code"])
    report(f"{year}: 2A and 2B present", {"2A", "2B"} <= codes)
    report(f"{year}: no plain '20' code (Corsica must be split)", "20" not in codes)
    report(f"{year}: no numeric-looking code lost its leading zero (e.g. '1' instead of '01')",
           not any(re.fullmatch(r"\d", c) for c in codes))
    overseas = sorted(c for c in EXPECTED_DEPS[year] if c.startswith("97"))
    report(f"{year}: overseas départements present {overseas}", set(overseas) <= codes, sorted(set(overseas) - codes))

display(deps.loc[deps["dep_code"].isin(["2A", "2B"]) | deps["dep_type"].ne("metropole"),
                 ["year", "dep_code", "dep_name", "dep_type", "inscrits", "exprimes", "page_heading", "source_url"]])

[OK] 2002: 2A and 2B present
[OK] 2002: no plain '20' code (Corsica must be split)
[OK] 2002: no numeric-looking code lost its leading zero (e.g. '1' instead of '01')
[OK] 2002: overseas départements present ['971', '972', '973', '974']
[OK] 2022: 2A and 2B present
[OK] 2022: no plain '20' code (Corsica must be split)
[OK] 2022: no numeric-looking code lost its leading zero (e.g. '1' instead of '01')
[OK] 2022: overseas départements present ['971', '972', '973', '974', '976']


,year,dep_code,dep_name,dep_type,inscrits,exprimes,page_heading,source_url
448,2002,2A,Corse-du-Sud,metropole,86526,50261,CORSE SUD (2A) (résultats officiels),https://www.archives-resultats-elections.inter...
464,2002,2B,Haute-Corse,metropole,104867,59201,HAUTE CORSE (2B) (résultats officiels),https://www.archives-resultats-elections.inter...
1536,2002,971,Guadeloupe,DOM,281550,91907,GUADELOUPE (971) (résultats officiels),https://www.archives-resultats-elections.inter...
1552,2002,972,Martinique,DOM,266629,89428,MARTINIQUE (972) (résultats officiels),https://www.archives-resultats-elections.inter...
1568,2002,973,Guyane,DOM,51787,23664,GUYANE (973) (résultats officiels),https://www.archives-resultats-elections.inter...
1584,2002,974,La Réunion,DOM,436889,233721,LA REUNION (974) (résultats officiels),https://www.archives-resultats-elections.inter...
1600,2002,975,Saint-Pierre-et-Miquelon,COM,4813,1942,SAINT PIERRE ET MIQUELON (975) (résultats offi...,https://www.archives-resultats-elections.inter...
1616,2002,976,Mayotte,DOM,52218,21189,MAYOTTE (976) (résultats officiels),https://www.archives-resultats-elections.inter...
1632,2002,986,Wallis-et-Futuna,COM,9353,5990,WALLIS ET FUTUNA (986) (résultats officiels),https://www.archives-resultats-elections.inter...
1648,2002,987,Polynésie française,COM,151505,75559,POLYNESIE FRANCAISE (987) (résultats officiels),https://www.archives-resultats-elections.inter...


### Pages correspond to the expected département

Compares the département name with the page heading, and the code with the code found in the URL.
Mismatches are not necessarily errors (e.g. overseas region pages, headings without the name) but
should be looked at.

In [129]:
name_ok = deps.apply(lambda r: norm_dep_name(r["dep_name"]) in norm_text(r["page_heading"]), axis=1)
report("département name appears in the page heading", name_ok.all(),
       deps.loc[~name_ok, ["year", "dep_code", "dep_name", "page_heading", "source_url"]])

urls = pd.concat([dep_urls_2002, dep_urls_2022], ignore_index=True)
urls["code_in_url"] = urls["url"].map(code_from_url)
report("code in URL matches département code", urls["code_in_url"].eq(urls["dep_code"]).all(),
       urls.loc[urls["code_in_url"].ne(urls["dep_code"]), ["year", "dep_code", "dep_name", "code_in_url", "url", "method"]])

[CHECK] département name appears in the page heading


,year,dep_code,dep_name,page_heading,source_url
448,2002,2A,Corse-du-Sud,CORSE SUD (2A) (résultats officiels),https://www.archives-resultats-elections.inter...
1680,2022,01,Ain,Les archives des élections en France,https://www.archives-resultats-elections.inter...
1692,2022,02,Aisne,Les archives des élections en France,https://www.archives-resultats-elections.inter...
1704,2022,03,Allier,Les archives des élections en France,https://www.archives-resultats-elections.inter...
1716,2022,04,Alpes-de-Haute-Provence,Les archives des élections en France,https://www.archives-resultats-elections.inter...
...,...,...,...,...,...
2916,2022,986,Wallis-et-Futuna,Les archives des élections en France,https://www.archives-resultats-elections.inter...
2928,2022,987,Polynésie française,Les archives des élections en France,https://www.archives-resultats-elections.inter...
2940,2022,988,Nouvelle-Calédonie,Les archives des élections en France,https://www.archives-resultats-elections.inter...
2952,2022,ZX,Saint-Martin/Saint-Barthélemy,Les archives des élections en France,https://www.archives-resultats-elections.inter...


[CHECK] code in URL matches département code


,year,dep_code,dep_name,code_in_url,url,method
211,2022,ZX,Saint-Martin/Saint-Barthélemy,977,https://www.archives-resultats-elections.inter...,link label


## 12. Save outputs

UTF-8 CSVs. When reading them back, keep département codes as text:
`pd.read_csv(path, dtype={"dep_code": str})` (otherwise `01` becomes `1`).

In [130]:
raw_2002 = results[results["year"] == 2002]
raw_2022 = results[results["year"] == 2022]

raw_2002.to_csv(RAW_DIR / "presidential_2002_departments.csv", index=False, encoding="utf-8")
raw_2022.to_csv(RAW_DIR / "presidential_2022_departments.csv", index=False, encoding="utf-8")

processed = results.drop(columns=["source_url", "page_heading", "table_round_label", "turnout_matches_sum"])
processed.to_csv(PROCESSED_DIR / "presidential_departments_2002_2022.csv", index=False, encoding="utf-8")

failures = pd.concat([failures_2002, failures_2022], ignore_index=True)
if not failures.empty:
    failures.to_csv(RAW_DIR / "presidential_departments_failures.csv", index=False, encoding="utf-8")

print("2002:", raw_2002.shape, "| 2022:", raw_2022.shape, "| combined:", processed.shape, "| failures:", len(failures))

2002: (1680, 20) | 2022: (1296, 20) | combined: (2976, 16) | failures: 0


In [131]:
# Quick re-read check (accents and codes preserved)
check = pd.read_csv(PROCESSED_DIR / "presidential_departments_2002_2022.csv", dtype={"dep_code": str}, encoding="utf-8")
check[check["dep_code"].isin(["01", "2A", "2B", "971"])].groupby(["year", "dep_code", "dep_name"]).size()

year  dep_code  dep_name    
2002  01        Ain             16
      2A        Corse-du-Sud    16
      2B        Haute-Corse     16
      971       Guadeloupe      16
2022  01        Ain             12
      2A        Corse-du-Sud    12
      2B        Haute-Corse     12
      971       Guadeloupe      12
dtype: int64